<a href="https://colab.research.google.com/github/domore9630-hue/PINNs-Physics-AI-Solutions/blob/main/01_PINNs_Intro_PyTorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available (GPU): {torch.cuda.is_available()}")

PyTorch Version: 2.11.0+cpu
CUDA Available (GPU): False


In [3]:
# بناء شبكة عصبية بسيطة تتكون من طبقة مدخلات وطبقتين خفيتين ومخرج واحد
class PINN(nn.Module):
    def __init__(self):
        super(PINN, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 20),
            nn.Tanh(),
            nn.Linear(20, 20),
            nn.Tanh(),
            nn.Linear(20, 1)
        )

    def forward(self, x):
        return self.net(x)

# تجربة الشبكة العصبية بمدخلات عشوائية
model = PINN()
test_input = torch.tensor([[0.5]])
test_output = model(test_input)

print("PINN Model Architecture:")
print(model)
print(f"\nTest Output for Input 0.5: {test_output.item():.4f}")

PINN Model Architecture:
PINN(
  (net): Sequential(
    (0): Linear(in_features=1, out_features=20, bias=True)
    (1): Tanh()
    (2): Linear(in_features=20, out_features=20, bias=True)
    (3): Tanh()
    (4): Linear(in_features=20, out_features=1, bias=True)
  )
)

Test Output for Input 0.5: -0.3042


In [6]:
# فكرة لحساب خسارة الفيزياء لـ معادلة تفاضلية: du/dt + u = 0
import torch

def physics_loss(model, t):
    t.requires_grad_(True)
    u = model(t)
    # حساب المشتقة du/dt تلقائياً
    u_t = torch.autograd.grad(u, t, grad_outputs=torch.ones_like(u), create_graph=True)[0]

    # بقايا المعادلة الفيزيائية (Residual)
    physics_residual = u_t + u

    # حساب خسارة الفيزياء (يجب أن تقترب من الصفر)
    return torch.mean(physics_residual**2)